# Ngày 2: Phân loại lớp phủ Hà Nội (LULC) bằng Random Forest trên GEE

Notebook này thực hiện quy trình phân loại lớp phủ bề mặt (Land Use / Land Cover - LULC) cho Thành phố Hà Nội sử dụng ảnh vệ tinh **Sentinel-2** và thuật toán học máy **Random Forest** trên nền tảng đám mây **Google Earth Engine (GEE)**.

### Nhiệm vụ thực hiện:
1. **Thiết lập tập mẫu huấn luyện (Training data)** cho 4 lớp chính: Đô thị (Urban), Nước (Water), Nông nghiệp (Agriculture), và Cây xanh (Vegetation/Greenery).
2. **Truy vấn và tiền xử lý ảnh vệ tinh Sentinel-2** trong năm 2024 (lọc mây và tạo ảnh composite median).
3. **Huấn luyện mô hình Random Forest** trên GEE để phân loại lớp phủ toàn thành phố.
4. **Tính toán và thống kê diện tích** từng lớp phủ (đặc biệt là đất xây dựng - Đô thị) cho từng Quận/Huyện của Hà Nội.
5. **So sánh diện tích đất xây dựng** giữa quận Cầu Giấy và quận Hà Đông.
6. **Trực quan hóa bản đồ LULC** tương tác.

## 1. Thiết lập môi trường và khởi tạo GEE

In [2]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import folium

# Khởi tạo Google Earth Engine
try:
    ee.Initialize(project='crested-library-500309-i2')
    print("GEE Initialized Successfully!")
except Exception as e:
    print("GEE initialization failed. Running Authenticate...")
    ee.Authenticate()
    ee.Initialize(project='crested-library-500309-i2')
    print("GEE Authenticated & Initialized Successfully!")

GEE Initialized Successfully!


## 2. Tải và lọc ranh giới hành chính Hà Nội

Chúng ta sẽ sử dụng dữ liệu ranh giới hành chính cấp huyện từ Open Development Mekong để thống kê cho Hà Nội.

In [3]:
geojson_url = "https://data.opendevelopmentmekong.net/dataset/6f054351-bf2c-422e-8deb-0a511d63a315/resource/78b3fb31-8c96-47d3-af64-d1a6e168e2ea/download/diaphanhuyen.geojson"

print("Loading boundary GeoJSON...")
gdf = gpd.read_file(geojson_url)
hanoi_gdf = gdf[gdf['Ten_Tinh'] == 'Hà Nội'].copy()
print(f"Loaded {len(hanoi_gdf)} districts/towns for Hanoi.")

# Chuyển đổi sang FeatureCollection của GEE
hanoi_fc = geemap.gdf_to_ee(hanoi_gdf)

Loading boundary GeoJSON...
Loaded 29 districts/towns for Hanoi.


## 3. Truy vấn và xử lý ảnh Sentinel-2

Chúng ta sẽ truy vấn ảnh vệ tinh Sentinel-2 (lớp Surface Reflectance) cho năm 2024, lọc mây bằng kênh `QA60`, chọn 10 ảnh sạch mây nhất và tạo ảnh Median Composite cắt theo ranh giới Hà Nội.

In [4]:
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask)

print("Querying Sentinel-2 ImageCollection...")
s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(hanoi_fc)
          .filterDate('2024-01-01', '2024-12-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

# Lọc lấy 10 ảnh sạch mây nhất để tối ưu hóa hiệu suất và tránh lỗi tràn bộ nhớ
s2_clean = s2_col.sort('CLOUDY_PIXEL_PERCENTAGE').limit(10)

# Tạo ảnh median composite và chọn các kênh phổ chính
# B2 (Blue), B3 (Green), B4 (Red), B8 (NIR), B11 (SWIR1), B12 (SWIR2)
base_image = s2_clean.map(maskS2clouds).median().select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']).clip(hanoi_fc)

# Tính toán các chỉ số phổ phụ trợ
ndvi = base_image.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi = base_image.normalizedDifference(['B11', 'B8']).rename('NDBI')
mndwi = base_image.normalizedDifference(['B3', 'B11']).rename('MNDWI')

# Gộp các băng chỉ số phổ vào ảnh composite gốc
image = base_image.addBands([ndvi, ndbi, mndwi])
print("Sentinel-2 Composite with NDVI, NDBI, MNDWI created successfully!")

Querying Sentinel-2 ImageCollection...
Sentinel-2 Composite with NDVI, NDBI, MNDWI created successfully!


## 4. Thiết lập tập mẫu huấn luyện dạng Vùng (Polygons)

Chúng ta xác định tọa độ các vùng đa giác (Polygons) đại diện cho 4 lớp phủ chính tại các khu vực địa lý khác nhau của Hà Nội:
- **Lớp 0: Nước (Water)**: Hồ Tây, Sông Hồng (Phía Bắc & Phía Nam), Hồ Suối Hai (Ba Vì).
- **Lớp 1: Đô thị (Urban)**: Khu dân cư Hoàn Kiếm, nhà cao tầng Cầu Giấy, đô thị mới Hà Đông, Thị xã Sơn Tây.
- **Lớp 2: Nông nghiệp (Agriculture)**: Vùng sản xuất hoa màu ở Ba Vì, Sóc Sơn, Mỹ Đức, Đông Anh.
- **Lớp 3: Cây xanh (Vegetation/Greenery)**: Rừng phòng hộ Sóc Sơn, Vườn quốc gia Ba Vì, khu rừng núi đá vôi Hương Sơn (Mỹ Đức).

In [5]:
polygons_data = {
    0: { # Water
        'West Lake': [[105.80, 21.05], [105.83, 21.05], [105.83, 21.08], [105.80, 21.08]],
        'Red River South': [[105.86, 21.03], [105.89, 21.03], [105.89, 21.07], [105.86, 21.07]],
        'Red River North': [[105.78, 21.10], [105.82, 21.10], [105.82, 21.13], [105.78, 21.13]],
        'Suoi Hai Lake': [[105.34, 21.13], [105.37, 21.13], [105.37, 21.16], [105.34, 21.16]]
    },
    1: { # Urban
        'Hoan Kiem': [[105.84, 21.02], [105.86, 21.02], [105.86, 21.04], [105.84, 21.04]],
        'Cau Giay': [[105.78, 21.01], [105.80, 21.01], [105.80, 21.04], [105.78, 21.04]],
        'Ha Dong': [[105.76, 20.96], [105.78, 20.96], [105.78, 20.98], [105.76, 20.98]],
        'Son Tay Town': [[105.50, 21.12], [105.52, 21.12], [105.52, 21.14], [105.50, 21.14]]
    },
    2: { # Agriculture
        'Ba Vi Rural': [[105.38, 21.18], [105.42, 21.18], [105.42, 21.21], [105.38, 21.21]],
        'Soc Son Rural': [[105.83, 21.23], [105.87, 21.23], [105.87, 21.26], [105.83, 21.26]],
        'My Duc Plains': [[105.72, 20.73], [105.76, 20.73], [105.76, 20.76], [105.72, 20.76]],
        'Dong Anh Crop': [[105.83, 21.12], [105.86, 21.12], [105.86, 21.15], [105.83, 21.15]]
    },
    3: { # Greenery
        'Soc Son Hills': [[105.79, 21.27], [105.82, 21.27], [105.82, 21.30], [105.79, 21.30]],
        'Ba Vi National Park': [[105.35, 21.03], [105.38, 21.03], [105.38, 21.07], [105.35, 21.07]],
        'Huong Son Hills': [[105.73, 20.61], [105.76, 20.61], [105.76, 20.64], [105.73, 20.64]]
    }
}
print("Polygons successfully defined!")

Polygons successfully defined!


## 5. Trích xuất giá trị phổ bằng phương pháp lấy mẫu phân tầng (Stratified Sampling) trong các Vùng đại diện

Để lấy mẫu hiệu quả và ổn định từ GEE, ta gộp các đa giác huấn luyện vào một FeatureCollection và gán thuộc tính `class`. Sau đó, ta vẽ các đa giác này lên một ảnh trống để tạo băng nhãn `class` (các khu vực ngoài đa giác sẽ bị ẩn/null). Bằng cách sử dụng phương thức `.stratifiedSample()`, GEE sẽ tự động lấy mẫu phân tầng ngẫu nhiên **150 pixel cho mỗi lớp phủ** tại độ phân giải 10m của Sentinel-2, tạo nên tổng cộng **600 mẫu huấn luyện** phân bố đều khắp Hà Nội.

In [6]:
print("Creating training polygons FeatureCollection...")
features_list = []
for class_val, polys in polygons_data.items():
    for name, coords in polys.items():
        geom = ee.Geometry.Polygon(coords)
        features_list.append(ee.Feature(geom, {'class': class_val}))

training_polygons = ee.FeatureCollection(features_list)

# Vẽ đa giác lên ảnh để tạo băng nhãn lớp phủ
class_band = ee.Image().byte().paint(training_polygons, 'class').rename('class')

# Gộp băng nhãn vào ảnh composite chứa các chỉ số phổ
image_with_class = image.addBands(class_band)

print("Performing stratified sampling across the polygons...")
training = image_with_class.stratifiedSample(
    numPoints=150,
    classBand='class',
    region=training_polygons.geometry(),
    scale=10,
    seed=42,
    geometries=True
)

total_samples = training.size().getInfo()
print(f"Total training pixels sampled from polygons: {total_samples}")

Creating training polygons FeatureCollection...
Performing stratified sampling across the polygons...
Total training pixels sampled from polygons: 600


## 6. Huấn luyện mô hình Random Forest

Ta chạy mô hình Random Forest Classifier với 100 cây quyết định trên tập mẫu huấn luyện lớn vừa thu thập được.

In [7]:
print("Training Random Forest Classifier on GEE (100 trees)...")
classifier = ee.Classifier.smileRandomForest(100).train(
    features=training,
    classProperty='class',
    inputProperties=['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDBI', 'MNDWI']
)

print("Classifying the Sentinel-2 image...")
classified = image.classify(classifier)
print("Image classification complete!")

Training Random Forest Classifier on GEE (100 trees)...
Classifying the Sentinel-2 image...
Image classification complete!


## 7. Đánh giá độ chính xác và Đề xuất cải thiện

### Đánh giá độ chính xác (Accuracy Assessment)
Để đánh giá khả năng khớp mẫu của mô hình trên tập dữ liệu huấn luyện, ta tính toán Ma trận nhầm lẫn (Confusion Matrix), Độ chính xác toàn cục (Overall Accuracy) và Hệ số Kappa từ tập mẫu huấn luyện (Resubstitution Accuracy).

### Đề xuất phương pháp nâng cao độ chính xác phân loại:
1. **Tăng quy mô tập mẫu (Training Samples)**:
   - Tăng số lượng điểm mẫu (từ 50-100 điểm cho mỗi lớp) phủ khắp các vùng địa lý của Hà Nội để mô hình học được sự đa dạng của các đối tượng địa lý.
   - Thay vì chỉ sử dụng các điểm đơn lẻ, có thể sử dụng các vùng đa giác nhỏ (Polygons) để lấy mẫu nhiều pixel hơn, tăng tính đại diện.
2. **Bổ sung các chỉ số phổ (Spectral Indices)**:
   - Đưa thêm các chỉ số tính toán từ ảnh Sentinel-2 làm biến đầu vào (Features) cho mô hình (tương tự như Ngày 1), ví dụ:
     - **NDVI** (Chỉ số thực vật): Giúp phân biệt tốt hơn giữa Cây xanh và Nông nghiệp/Đất trống.
     - **NDBI** (Chỉ số đất xây dựng) / **MNDWI** (Chỉ số nước cải tiến): Tăng cường khả năng nhận diện các khu đô thị và mặt nước.
3. **Tối ưu hóa siêu tham số (Hyperparameter Tuning)**:
   - Tăng số lượng cây quyết định trong Random Forest (`numberOfTrees` từ 30 lên 100 hoặc 150).
   - Điều chỉnh độ sâu tối đa của cây (`maxNodes`) hoặc số lượng thuộc tính tối thiểu ở mỗi lá để tránh hiện tượng overfitting.
4. **Chọn ảnh Composite theo mùa (Seasonal Compositing)**:
   - Thay vì median composite cho cả năm, ta nên tạo ảnh composite cho các tháng mùa khô (tháng 10 - tháng 3 năm sau) khi bầu trời ít mây và các ruộng nông nghiệp/cây xanh có sự tương phản phổ rõ rệt nhất.

In [8]:
# Tính confusion matrix cho tập huấn luyện
train_accuracy_obj = classifier.confusionMatrix()
accuracy = train_accuracy_obj.accuracy().getInfo()
kappa = train_accuracy_obj.kappa().getInfo()
conf_matrix = train_accuracy_obj.getInfo()

print("=== KẾT QUẢ ĐÁNH GIÁ ĐỘ CHÍNH XÁC (TẬP HUẤN LUYỆN) ===")
print(f"Độ chính xác toàn cục (Overall Accuracy): {accuracy * 100:.2f}%")
print(f"Hệ số Kappa (Kappa Coefficient): {kappa:.4f}")
print("\nMa trận nhầm lẫn (Confusion Matrix):")
for row in conf_matrix:
    print(row)

=== KẾT QUẢ ĐÁNH GIÁ ĐỘ CHÍNH XÁC (TẬP HUẤN LUYỆN) ===
Độ chính xác toàn cục (Overall Accuracy): 99.50%
Hệ số Kappa (Kappa Coefficient): 0.9933

Ma trận nhầm lẫn (Confusion Matrix):
[148, 1, 1, 0]
[0, 150, 0, 0]
[0, 0, 150, 0]
[1, 0, 0, 149]


## 8. Tính toán diện tích các lớp phủ của từng Quận/Huyện

Chúng ta sẽ tạo một ảnh 4 băng, mỗi băng biểu thị diện tích (m2) của một lớp phủ cụ thể bằng cách sử dụng `ee.Image.pixelArea()` kết hợp với mask của từng lớp phân loại. Sau đó dùng `reduceRegions` để tính tổng diện tích từng lớp cho các Quận/Huyện.

In [9]:
print("Calculating area images for all 4 classes...")
pixel_area = ee.Image.pixelArea()

class_areas = ee.Image.cat([
    pixel_area.updateMask(classified.eq(0)).rename('Water_Area'),
    pixel_area.updateMask(classified.eq(1)).rename('Urban_Area'),
    pixel_area.updateMask(classified.eq(2)).rename('Agri_Area'),
    pixel_area.updateMask(classified.eq(3)).rename('Green_Area')
])

print("Reducing regions to get area statistics per district (scale=10)...")
district_areas = class_areas.reduceRegions(
    collection=hanoi_fc,
    reducer=ee.Reducer.sum(),
    scale=10
)

print("Fetching statistics from GEE...")
results = district_areas.getInfo()
print("Data fetched successfully!")

Calculating area images for all 4 classes...
Reducing regions to get area statistics per district (scale=10)...
Fetching statistics from GEE...
Data fetched successfully!


## 9. Xử lý dữ liệu và xuất kết quả ra bảng CSV/Dataframe

In [10]:
rows = []
for feat in results['features']:
    props = feat['properties']
    dist_name = props.get('Ten_Huyen', 'Unknown')
    code_vung = props.get('Code_vung', 'Unknown')
    
    water_m2 = props.get('Water_Area', 0) or 0
    urban_m2 = props.get('Urban_Area', 0) or 0
    agri_m2 = props.get('Agri_Area', 0) or 0
    green_m2 = props.get('Green_Area', 0) or 0
    
    rows.append({
        'Code_Vung': code_vung,
        'District_Name': dist_name,
        'Water_Area_km2': water_m2 / 1e6,
        'Urban_Area_km2': urban_m2 / 1e6,
        'Agriculture_Area_km2': agri_m2 / 1e6,
        'Greenery_Area_km2': green_m2 / 1e6,
        'Total_LULC_Area_km2': (water_m2 + urban_m2 + agri_m2 + green_m2) / 1e6
    })

df_result = pd.DataFrame(rows).sort_values(by='Urban_Area_km2', ascending=False)

# Xuất bảng thống kê diện tích ra file CSV
output_csv = 'hanoi_lulc_district_areas.csv'
df_result.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"Exported area statistics to '{output_csv}' successfully!")
print("\nTable of results (in km2):")
print(df_result.to_string(index=False))

Exported area statistics to 'hanoi_lulc_district_areas.csv' successfully!

Table of results (in km2):
Code_Vung District_Name  Water_Area_km2  Urban_Area_km2  Agriculture_Area_km2  Greenery_Area_km2  Total_LULC_Area_km2
    01017      Dong Anh       43.384423       44.260503             94.732538           8.084286           190.461750
    01016       Soc Son       59.202457       34.231056            156.948050          53.667174           304.048738
    01271         Ba Vi       92.117386       31.938401            155.109674         140.003712           419.169172
    01018       Gia Lam       30.377598       31.538684             47.637959           8.833339           118.387580
    01019       Tu Liem       19.860654       31.038366             21.015650           2.802489            74.717159
    01277     Chuong My       42.565332       29.368594            129.031956          28.335108           229.300990
    01279    Thuong Tin       28.745285       28.925434             77.5

## 10. Trực quan hóa bản đồ tương tác lớp phủ LULC

Sử dụng thư viện `geemap` để hiển thị bản đồ LULC phân loại với màu sắc quy ước:
- Màu lam (Blue): Lớp Nước (`#0000FF`)
- Màu đỏ (Red): Lớp Đô thị (`#FF0000`)
- Màu vàng (Yellow): Lớp Nông nghiệp (`#FFFF00`)
- Màu xanh lục (Green): Lớp Cây xanh (`#008000`)

In [13]:
Map = geemap.Map(lite_mode=True)
Map.centerObject(hanoi_fc, 10)

# Thiết lập màu sắc trực quan
lulc_vis = {
    'min': 0,
    'max': 3,
    'palette': ['0000ff', 'ff0000', 'ffff00', '008000'] # Water, Urban, Agri, Green
}

# Thêm ranh giới huyện
Map.addLayer(hanoi_fc, {'color': 'black', 'fillColor': '00000000'}, 'Hanoi Districts')

# Thêm lớp LULC đã phân loại
Map.addLayer(classified, lulc_vis, 'Hanoi LULC 2024')

# Hiển thị bản đồ
Map

Map(center=[20.999184677179052, 105.69885038571539], controls=(WidgetControl(options=['position', 'transparent…